# 03 — Vulnerabilidade e priorização

Integramos epidemiologia e condições socioeconômicas. Associação não implica causalidade.

In [ ]:
from pathlib import Path
import pandas as pd, seaborn as sns, matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
ROOT=Path.cwd().parent if Path.cwd().name=="notebooks" else Path.cwd()
# Este módulo permanece integralmente demonstrativo até a integração da renda oficial.
casos=pd.read_csv(ROOT/"data"/"demo"/"dengue_semanal_demo.csv")
mun=pd.read_csv(ROOT/"data"/"demo"/"municipios_demo.csv")
FONTE="base sintética de demonstração (módulo socioeconômico)"
print("Fonte selecionada:",FONTE)
base=casos.groupby("municipio",as_index=False).casos.sum().merge(mun,on="municipio")
base["incidencia_100k"]=100000*base.casos/base.populacao
base["pressao_demo"]=0.65*(base.incidencia_100k/base.incidencia_100k.max())+0.35*base.vulnerabilidade_demo
base.sort_values("pressao_demo",ascending=False)

In [ ]:
sns.regplot(data=base,x="renda_pc_demo",y="incidencia_100k",scatter_kws={"s":90})
for _,r in base.iterrows(): plt.text(r.renda_pc_demo+12,r.incidencia_100k,r.municipio,fontsize=8)
plt.title("Renda e incidência: associação exploratória"); plt.xlabel("Renda per capita (demo)"); plt.ylabel("Incidência por 100 mil"); plt.show()

In [ ]:
X=StandardScaler().fit_transform(base[["incidencia_100k","vulnerabilidade_demo"]])
base["grupo_prioridade"]=KMeans(n_clusters=3,random_state=42,n_init=20).fit_predict(X)
sns.scatterplot(data=base,x="vulnerabilidade_demo",y="incidencia_100k",hue="grupo_prioridade",size="populacao",sizes=(80,650),palette="viridis")
for _,r in base.iterrows(): plt.text(r.vulnerabilidade_demo+.008,r.incidencia_100k,r.municipio,fontsize=8)
plt.title("Agrupamento exploratório para discutir prioridades"); plt.show()

In [ ]:
painel=base.sort_values("pressao_demo",ascending=False)[["municipio","casos","incidencia_100k","renda_pc_demo","vulnerabilidade_demo","pressao_demo","grupo_prioridade"]]
painel.style.background_gradient(subset=["incidencia_100k","vulnerabilidade_demo","pressao_demo"],cmap="YlOrRd").format({"incidencia_100k":"{:.1f}","vulnerabilidade_demo":"{:.2f}","pressao_demo":"{:.2f}"})

## Fechamento

1. Que municípios merecem investigação primeiro?
2. Quais recursos seriam mobilizados?
3. Que informação falta antes de decidir?
4. Como documentar critérios para que a decisão seja auditável?

> Modelos exploratórios apoiam perguntas; não substituem protocolos, conhecimento territorial ou avaliação causal.